In [0]:
# Dropdown widget for environment selection
dbutils.widgets.dropdown(
    name="environment",
    defaultValue="dev",
    choices=["dev", "prd", "qa"],
    label="select Environment"
)

# Get selected environment
env = dbutils.widgets.get("environment")

# Define table names dynamically for Silver and Gold layers
goldTablName   = f"saleslake_{env}.gold_{env}.RefinedSales"
silverTablName = f"saleslake_{env}.silver_{env}.cleanedsales"

In [0]:
# MERGE from Silver → Gold (SCD Type 1 logic)
spark.sql(f"""
MERGE INTO {goldTablName}  AS tgt
USING (SELECT * FROM  {silverTablName}
          WHERE ingest_ts > (
                      SELECT coalesce(MAX(last_updt_ts),to_timestamp('1990-01-01','yyyy-MM-dd')) 
                      FROM {goldTablName} )
                       )AS src
ON tgt.sale_id=src.sale_id
-- When record exists but something changed --> UPDATE (SCD Type 1)
WHEN MATCHED AND (
         tgt.product <> src.product  OR
         tgt.category <> src.category OR
         tgt.quantity <> src.quantity OR
         tgt.price <> src.price    OR
         tgt.sale_date <> src.sale_date OR
         tgt.region <> src.region
)
THEN UPDATE SET
    tgt.product = src.product,
    tgt.category = src.category,
    tgt.quantity = src.quantity,
    tgt.price = src.price,
    tgt.sale_date = src.sale_date,
    tgt.region = src.region,
    tgt.last_updt_ts = current_timestamp()

WHEN NOT MATCHED THEN INSERT (
    sale_id, product, category, quantity, price, sale_date, region,
    intial_load_ts,
    last_updt_ts
)
VALUES (
    src.sale_id,
    src.product,
    src.category,
    src.quantity,
    src.price,
    src.sale_date,
    src.region,
    current_timestamp(),
    current_timestamp()
)
""");